# 🌊 AquaGuard AI — Day 16: Multi-Step Forecasting Baselines
- **Date:** 04 Sep 2026
- **Phase:** Phase 3: Modeling & Validation
- **Objective:** Benchmark Persistence Forecaster against Multi-Step Ridge Regression and Random Forest on chronological holdout.
- **Inputs Consumed:** `data/processed/dataset_v2_features.parquet`, `configs/config.yaml`
- **Outputs Produced:** `artifacts/day16/forecasting_baselines.json`, `artifacts/day16/forecast_vs_actual.png`
- **Deliverable Satisfied:** Persistence baseline beat by multi-step models; MAE/RMSE holdout comparison.

In [1]:
import os, sys
from pathlib import Path
project_root = Path("..").resolve() if Path("..").resolve().name == "Water Potability" else Path(".").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)

import numpy as np
import pandas as pd
from src.utils import set_seed, load_config, ensure_dir
from src.io import load_parquet, save_artifact
from src.forecasting import create_forecasting_dataset, PersistenceForecaster, LinearRegressionForecaster, RandomForestForecaster
from src.evaluation import compute_forecasting_metrics
from src.plotting import plot_predicted_vs_actual

set_seed(42)
config = load_config("configs/config.yaml")
ensure_dir("artifacts/day16")

feat_df = load_parquet("data/processed/dataset_v2_features.parquet", required_by_day="Day 13")
target = config["track_a"]["target_column"]
feature_cols = [c for c in feat_df.columns if c != config["track_a"]["timestamp_column"]]
print(f"Forecasting Target: '{target}', Horizon: 6 steps ahead (6 hours)")

Forecasting Target: 'dissolved_oxygen', Horizon: 6 steps ahead (6 hours)


## 1. Construct Supervised Forecasting Dataset
Create (X_t, Y_{t+1...t+6}) pairs and perform chronological 80/20 train/validation split.

In [2]:
X_df, Y_df = create_forecasting_dataset(
    feat_df,
    feature_cols=feature_cols,
    target_col=target,
    horizon=config["track_a"]["forecast_horizon_steps"]
)

split_idx = int(len(X_df) * 0.80)
X_train, X_val = X_df.iloc[:split_idx], X_df.iloc[split_idx:]
Y_train, Y_val = Y_df.iloc[:split_idx], Y_df.iloc[split_idx:]
print(f"Forecasting split: Train={X_train.shape}, Val={X_val.shape}")

Forecasting split: Train=(484, 148), Val=(122, 148)


## 2. Evaluate Baseline Models: Persistence vs Ridge vs Random Forest
Compute MAE, RMSE, and Directional Accuracy across the 6-hour prediction horizon.

In [3]:
results = {}

# 1. Persistence Baseline
target_idx = feature_cols.index(target)
persist = PersistenceForecaster(target_col_idx=target_idx, horizon=6)
persist.fit(X_train)
y_pred_persist = persist.predict(X_val)
results["Persistence_Baseline"] = compute_forecasting_metrics(Y_val.values, y_pred_persist)

# 2. Multi-Step Ridge Regression
ridge = LinearRegressionForecaster(alpha=1.0, random_state=config["seed"])
ridge.fit(X_train, Y_train)
y_pred_ridge = ridge.predict(X_val)
results["MultiStep_Ridge"] = compute_forecasting_metrics(Y_val.values, y_pred_ridge)

# 3. Multi-Step Random Forest
rf = RandomForestForecaster(n_estimators=100, max_depth=6, random_state=config["seed"])
rf.fit(X_train, Y_train)
y_pred_rf = rf.predict(X_val)
results["MultiStep_RandomForest"] = compute_forecasting_metrics(Y_val.values, y_pred_rf)

save_artifact(results, "artifacts/day16/forecasting_baselines.json")
print("Forecasting Baseline Metrics:")
for k, v in results.items():
    print(f"  {k}: {v}")

Forecasting Baseline Metrics:
  Persistence_Baseline: {'mae': 0.6424, 'rmse': 0.795, 'mape_pct': 6.8, 'mda_pct': 0.0}
  MultiStep_Ridge: {'mae': 0.2973, 'rmse': 0.3802, 'mape_pct': 3.14, 'mda_pct': 66.89}
  MultiStep_RandomForest: {'mae': 0.2291, 'rmse': 0.3341, 'mape_pct': 2.43, 'mda_pct': 69.02}


## 3. Predicted vs Observed Trajectory Visuals
Compare lead step T+1 predictions against actual observations on validation holdout.

In [4]:
plot_predicted_vs_actual(
    Y_val.values,
    y_pred_rf,
    sensor_name=target,
    lead_step=1,
    title="Random Forest Forecaster vs Actual Observed",
    save_path="artifacts/day16/forecast_vs_actual.png"
)
print("Forecast vs actual plot saved to artifacts/day16/forecast_vs_actual.png")

Forecast vs actual plot saved to artifacts/day16/forecast_vs_actual.png


## 📌 Day 16 Execution Findings & Sign-Off
- **Key Finding:** Random Forest achieved lowest MAE, significantly outperforming Persistence.
- **Artifacts Saved:** `artifacts/day16/forecasting_baselines.json`, `artifacts/day16/forecast_vs_actual.png`.
- **Handoff:** Ready for Day 17 advanced model exploration and sequence optimization.